In [20]:
import kagglehub
import pandas as pd

In [50]:
path = kagglehub.dataset_download("vitaliymalcev/russian-passenger-air-service-20072020")
print("Path to dataset files:", path)

months = [
  'January',
  'February',
  'March',
  'April',
  'May',
  'June',
  'July',
  'August',
  'September',
  'October',
  'November',
  'December',
]

df_melted = df.melt(
    id_vars=['Airport name', 'Year'], # Фиксированные столбцы
    value_vars=months,                # Столбцы, которые нужно развернуть
    var_name='Month',                 # Название нового столбца для месяцев
    value_name='Quantity'             # Название нового столбца для значений
)

Path to dataset files: /kaggle/input/russian-passenger-air-service-20072020


In [94]:
month_mapping = {month: index + 1 for index, month in enumerate(months)}
X['Month']=X['Month'].map(month_mapping)
df_melted['Month'] = df_melted['Month'].map(month_mapping)

/tmp/ipython-input-94-3935225535.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Month']=X['Month'].map(month_mapping)


In [43]:
df_melted['Date'] = pd.to_datetime(df_melted['Year'].astype(str) + '-' + df_melted['Month'].astype(str) + '-01')
df_melted.sort_values(by='Date', inplace=True)

In [59]:
df_pulkovo = df_melted[df_melted['Airport name'] == 'Saint Petersburg (Pulkovo)']

In [60]:
df_pulkovo.reset_index(drop=True, inplace=True)

In [61]:
#Для оценки качества моделей я выберу Mean Absolute Error (MAE). Эта метрика показывает среднюю абсолютную ошибку между предсказанными и фактическими значениями, что позволяет лучше понять, насколько точны наши прогнозы. MAE менее чувствителен к выбросам по сравнению с RMSE, что делает его более подходящим для анализа временных рядов.

In [62]:
from sklearn.model_selection import train_test_split
train = df_pulkovo[df_pulkovo['Year'] < 2019]
test = df_pulkovo[df_pulkovo['Year'] >= 2019]

In [63]:
print(df_pulkovo.shape)
print(df_pulkovo.head())

(168, 4)
                 Airport name  Year    Month   Quantity
0  Saint Petersburg (Pulkovo)  2020  January  1328413.0
1  Saint Petersburg (Pulkovo)  2019  January  1230150.0
2  Saint Petersburg (Pulkovo)  2018  January  1079174.0
3  Saint Petersburg (Pulkovo)  2017  January   982522.0
4  Saint Petersburg (Pulkovo)  2016  January   784263.0


In [79]:
# Подготовка данных для модели
X = df_melted[['Year', 'Month']]
y = df_melted['Quantity']

In [95]:
from sklearn.model_selection import train_test_split

# Разделение на train и test с перемешиванием
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

# Проверка размерностей
print("Shape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Обучение бейзлайн модели
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

# Прогнозирование
y_pred_baseline = baseline_model.predict(X_test)

# Оценка качества модели
mse_baseline = mean_squared_error(y_test, y_pred_baseline)
print(f'Baseline MSE: {mse_baseline}')

Shape of X_train: (38025, 2)
Shape of y_train: (38025,)
Baseline MSE: 47655568683.55975


In [96]:
from sklearn.ensemble import RandomForestRegressor

# Обучение модели случайного леса
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Прогнозирование
y_pred_rf = rf_model.predict(X_test)

# Оценка качества
mae_rf = mean_absolute_error(y_test, y_pred_rf)
print(f'Random Forest MAE: {mae_rf}')

Random Forest MAE: 55761.39359245395


In [97]:
from sklearn.model_selection import GridSearchCV

# Определение параметров для поиска
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=5, scoring='neg_mean_absolute_error')
grid_search.fit(X_train, y_train)

# Лучшая модель
best_rf_model = grid_search.best_estimator_
y_pred_best_rf = best_rf_model.predict(X_test)

# Оценка качества
mae_best_rf = mean_absolute_error(y_test, y_pred_best_rf)
print(f'Best Random Forest MAE: {mae_best_rf}')

Best Random Forest MAE: 55714.09800773569
